## Task-1

In [4]:
import pandas as pd

# Load your dataset (update the filename if it's different)
df = pd.read_csv('train.csv')

# 1. Print the shape to confirm dimensions
print("Dataset Shape:", df.shape)

# 2. Check the data types and missing value counts
print("\n--- Dataset Info ---")
print(df.info(verbose=True, show_counts=True))

# 3. Look at the first 5 rows to see how features are named
print("\n--- First 5 Rows ---")
print(df.head())

Dataset Shape: (595212, 59)

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 595212 entries, 0 to 595211
Data columns (total 59 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              595212 non-null  int64  
 1   target          595212 non-null  int64  
 2   ps_ind_01       595212 non-null  int64  
 3   ps_ind_02_cat   595212 non-null  int64  
 4   ps_ind_03       595212 non-null  int64  
 5   ps_ind_04_cat   595212 non-null  int64  
 6   ps_ind_05_cat   595212 non-null  int64  
 7   ps_ind_06_bin   595212 non-null  int64  
 8   ps_ind_07_bin   595212 non-null  int64  
 9   ps_ind_08_bin   595212 non-null  int64  
 10  ps_ind_09_bin   595212 non-null  int64  
 11  ps_ind_10_bin   595212 non-null  int64  
 12  ps_ind_11_bin   595212 non-null  int64  
 13  ps_ind_12_bin   595212 non-null  int64  
 14  ps_ind_13_bin   595212 non-null  int64  
 15  ps_ind_14       595212 non-null  int64  
 16  ps_ind

In [5]:
import numpy as np

# --- Challenge 1: Handling Hidden Missing Values ---
# The dataset description notes that missing values are encoded as -1.
# We need to convert these to proper NaN (Not a Number) values so Pandas can handle them.

df.replace(-1, np.nan, inplace=True)

# Now let's see how many missing values we ACTUALLY have per column
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

print("--- Columns with Missing Values ---")
print(missing_data)
print(f"\nTotal columns with missing values: {len(missing_data)}")

# --- Challenge 2: Checking Class Imbalance ---
# Let's see how many people actually filed a claim versus how many didn't.

print("\n--- Target Variable Distribution ---")
claim_counts = df['target'].value_counts()
print(claim_counts)

# Calculate the percentage
claim_percentage = (claim_counts[1] / len(df)) * 100
print(f"\nPercentage of customers who filed a claim: {claim_percentage:.2f}%")

--- Columns with Missing Values ---
ps_car_03_cat    411231
ps_car_05_cat    266551
ps_reg_03        107772
ps_car_14         42620
ps_car_07_cat     11489
ps_ind_05_cat      5809
ps_car_09_cat       569
ps_ind_02_cat       216
ps_car_01_cat       107
ps_ind_04_cat        83
ps_car_02_cat         5
ps_car_11             5
ps_car_12             1
dtype: int64

Total columns with missing values: 13

--- Target Variable Distribution ---
target
0    573518
1     21694
Name: count, dtype: int64

Percentage of customers who filed a claim: 3.64%


In [6]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# --- 1. Handling Missing Data ---

# Drop columns with excessive missing values (>50%)
df.drop(['ps_car_03_cat'], axis=1, inplace=True)

# Separate remaining columns by type
cat_cols = [col for col in df.columns if '_cat' in col]
bin_cols = [col for col in df.columns if '_bin' in col]
# Continuous/Ordinal columns are the ones left over (excluding id and target)
cont_cols = [col for col in df.columns if col not in cat_cols + bin_cols + ['id', 'target']]

# Fill categorical missing values with the Mode
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Fill continuous missing values with the Median
for col in cont_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Double check that no missing values remain
print("Remaining missing values:", df.isnull().sum().sum())


# --- 2. Train-Test Split & Handling Imbalance ---

# Separate Features (X) and Target (y)
# We drop 'id' because it holds no predictive value
X = df.drop(['id', 'target'], axis=1)
y = df['target']

# Split the data FIRST to prevent data leakage (80% Train, 20% Test)
# stratify=y ensures the 3.64% ratio is maintained in both splits before SMOTE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nBefore SMOTE - Training target distribution:\n{y_train.value_counts()}")

# Apply SMOTE only to the Training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE - Training target distribution:\n{y_train_sm.value_counts()}")

C:\Users\KOLIPAKA LAVANYA\AppData\Local\Temp\ipykernel_19132\419938222.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
C:\Users\KOLIPAKA LAVANYA\AppData\Local\Temp\ipykernel_19132\419938222.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behave

Remaining missing values: 0

Before SMOTE - Training target distribution:
target
0    458814
1     17355
Name: count, dtype: int64

After SMOTE - Training target distribution:
target
0    458814
1    458814
Name: count, dtype: int64


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import time

# --- 1. Logistic Regression Model ---
print("Training Logistic Regression...")
start_time = time.time()

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_sm, y_train_sm)

# Predict on the UNSEEN test set
y_pred_log = log_reg.predict(X_test)
y_prob_log = log_reg.predict_proba(X_test)[:, 1] # Get probabilities for ROC-AUC

log_time = time.time() - start_time
print(f"Logistic Regression trained in {log_time:.2f} seconds.\n")


# --- 2. Random Forest Model ---
print("Training Random Forest (this may take a minute or two)...")
start_time = time.time()

# Using max_depth and n_jobs=-1 (uses all CPU cores) to keep training time reasonable
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

# Predict on the UNSEEN test set
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

rf_time = time.time() - start_time
print(f"Random Forest trained in {rf_time:.2f} seconds.\n")


# --- 3. Model Comparison Report ---
print("="*50)
print("             MODEL COMPARISON REPORT")
print("="*50)

print("\n--- Logistic Regression Performance ---")
print(classification_report(y_test, y_pred_log))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_log):.4f}")

print("\n--- Random Forest Performance ---")
print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.4f}")

Training Logistic Regression...


C:\Users\KOLIPAKA LAVANYA\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression trained in 161.96 seconds.

Training Random Forest (this may take a minute or two)...
Random Forest trained in 78.63 seconds.

             MODEL COMPARISON REPORT

--- Logistic Regression Performance ---
              precision    recall  f1-score   support

           0       0.96      0.92      0.94    114704
           1       0.04      0.10      0.06      4339

    accuracy                           0.89    119043
   macro avg       0.50      0.51      0.50    119043
weighted avg       0.93      0.89      0.91    119043

ROC-AUC Score: 0.5270

--- Random Forest Performance ---
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    114704
           1       0.05      0.06      0.05      4339

    accuracy                           0.93    119043
   macro avg       0.51      0.51      0.51    119043
weighted avg       0.93      0.93      0.93    119043

ROC-AUC Score: 0.5551
